In [2]:
import pandas as pd
import numpy as np
import glob
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from tqdm import tqdm

# ---------------------------------------------------------
# 1. LOAD TRAIN + TEST DATA FROM ALL 7 SITES
# ---------------------------------------------------------

train_files = sorted(glob.glob("./train_sets/site_*_train.csv"))
test_files  = sorted(glob.glob("./test_sets/site_*_test.csv"))

df_train = pd.concat([pd.read_csv(f) for f in train_files], ignore_index=True)
df_test  = pd.concat([pd.read_csv(f) for f in test_files], ignore_index=True)

print(f"Total training samples: {len(df_train)}")
print(f"Total testing samples:  {len(df_test)}")


# ---------------------------------------------------------
# 2. FEATURES AND TARGETS
# ---------------------------------------------------------

features = [
    "O3_forecast", "NO2_forecast", "T_forecast", "q_forecast",
    "u_forecast", "v_forecast", "w_forecast",
    "NO2_satellite", "HCHO_satellite", "ratio_satellite",
    "year", "month", "day", "hour"
]

targets = ["O3_target", "NO2_target"]

X_train = df_train[features]
y_train = df_train[targets]

X_test = df_test[features]
y_test = df_test[targets]


# ---------------------------------------------------------
# 3. TRAIN RANDOM FOREST WITH PROGRESS BAR
# ---------------------------------------------------------

n_trees = 500   # number of trees

print("\nTraining Random Forest on CPU...")
print("This may take a few minutes...\n")

# Prepare model
model = RandomForestRegressor(
    n_estimators=n_trees,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
    warm_start=True   # IMPORTANT: allows incremental training
)

# Train tree-by-tree with progress bar
for i in tqdm(range(1, n_trees + 1), desc="Training Progress"):
    model.n_estimators = i
    model.fit(X_train, y_train)


print("Training completed.")


# ---------------------------------------------------------
# 4. PREDICT & EVALUATE
# ---------------------------------------------------------

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n==============================")
print("   RANDOM FOREST PERFORMANCE  ")
print("==============================")
print(f"RMSE: {rmse:.3f}")
print(f"R²  : {r2:.3f}")


# ---------------------------------------------------------
# 5. SAVE TRAINED MODEL
# ---------------------------------------------------------

joblib.dump(model, "rf_all_sites.pkl")
print("\n✔ Saved model as rf_all_sites.pkl")


Total training samples: 120172
Total testing samples:  25752

Training Random Forest on CPU...
This may take a few minutes...



Training Progress: 100%|█████████████████████████████████████████████████████████████| 500/500 [08:11<00:00,  1.02it/s]


Training completed.

   RANDOM FOREST PERFORMANCE  
RMSE: 26.604
R²  : -0.490

✔ Saved model as rf_all_sites.pkl
